In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

df = pd.read_csv('./all_catapult_data_16_Jul_25.csv')

In [2]:
df.shape

(50046, 100)

In [3]:
# strip whitespace, replace spaces/slashes with underscores
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'\s+', '_', regex=True)
      .str.replace(r'[\/\(\)\-]', '', regex=True)
)

In [4]:

def clean_text(s):
    if pd.isnull(s):
        return s
    s = s.strip()                        # Remove leading/trailing whitespace
    s = re.sub(r'\s+', ' ', s)          # Replace multiple spaces with single
    s = s.lower()                       # Convert to lowercase (optional)
    # s = s.replace('_', ' ')             # Replace underscores with space
    s = s.title()                       # Convert to Title Case

    return s

# Apply cleaning to each text column
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)

In [5]:
#convert date to datetime format
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

In [6]:
#select only the 'Game' tag, exclude training etc 
# df = df[df['tags'] == 'Game']
df = df.drop('tags', axis=1)

In [7]:
df['duration'] = df['duration'] / 60 #convert duration to minutes for easy comparison

In [8]:
#select only the first and second halves ; ensure proper naming of splits
df = df[((df['split_name'] == '1St.Half') | (df['split_name'] == '2Nd.Half'))]
df['split_name'] = df['split_name'].str.replace('1St.Half', '1st Half')
df['split_name'] = df['split_name'].str.replace('2Nd.Half', '2nd Half')

In [9]:
# Select only session_title entries matching the pattern: 'Md1-Kcca Fc-Ura Fc-Home-League-Win' ( standard naming convention)
pattern = r'^Md\d+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+$'
df = df[df['session_title'].str.match(pattern, na=False)]

In [10]:
df.shape

(12946, 99)

In [11]:

# 1. Clean the session title
df['session_title'] = (
    df['session_title']
    .str.strip()
    .str.replace(r'\.+', '', regex=True)         # remove dots like "F.C."
    .str.replace(r'\s*-\s*', '-', regex=True)    # normalize spacing around hyphens
    .str.replace(r'\s+', ' ', regex=True)        # reduce internal whitespace
    .str.title()                                 # ensure proper capitalization
)

# 2. Extract session components: 6 fields + ignore trailing data
regex = re.compile(
    r'^(Md\d+)-'            # Matchday
    r'(.+?)-'               # Club For
    r'(.+?)-'               # Club Against
    r'(.+?)'                # Part 1: location/league/result
    r'[-\s]+(.+?)'          # Part 2: location/league/result
    r'[-\s]+(.+?)'          # Part 3: location/league/result
    r'(?:\s|$)',            # Ignore trailing info
    re.IGNORECASE
)
extracted = df['session_title'].str.extract(regex)

# 3. Assign temporary column names
extracted.columns = [
    'match_day',
    'club_for',
    'club_against',
    'part1',
    'part2',
    'part3'
]

# 4. Define sets to identify
location_set = {'Home', 'Away'}
league_set   = {'League', 'Cup'}
result_set   = {'Win', 'Loss', 'Draw'}

# 5. Assign actual values to correct columns
extracted['location'] = None
extracted['league']   = None
extracted['result']   = None

for i, row in extracted.iterrows():
    for val in [row['part1'], row['part2'], row['part3']]:
        if pd.isna(val): continue
        val_clean = val.strip().title()
        if val_clean in location_set:
            extracted.at[i, 'location'] = val_clean
        elif val_clean in league_set:
            extracted.at[i, 'league'] = val_clean
        elif val_clean in result_set:
            extracted.at[i, 'result'] = val_clean

# 6. Keep only fully matched and valid rows
valid = extracted.dropna(subset=['location', 'league', 'result']).copy()

# 7. Finalize merge
df = df.loc[valid.index].reset_index(drop=True)
valid = valid.drop(columns=['part1', 'part2', 'part3']).reset_index(drop=True)
df = pd.concat([df.reset_index(drop=True), valid], axis=1)


In [12]:
df.shape

(12946, 105)

In [13]:
# Extract player name, club, and position from 'player_name' column

# STEP 1: Fix incorrect entries where second separator is an underscore
# This converts 'First Last - Club_Position' → 'First Last - Club - Position'
def fix_player_name_format(name):
    if re.fullmatch(r'.+ - .+_.+', name):
        return re.sub(r'(.+ - .+?)_(.+)', r'\1 - \2', name)
    return name

df['player_name'] = df['player_name'].apply(fix_player_name_format)

# STEP 2: Extract the fields from the fixed format
player_regex = re.compile(
    r'^(.+?)\s*-\s*'     # Player name
    r'(.+?)\s*-\s*'      # Club
    r'(.+?)$'            # Position
)

player_cols = df['player_name'].str.extract(player_regex)
player_cols.columns = ['p_name', 'player_club_', 'player_position']

# STEP 3: Keep only rows where all 3 groups were found
valid = player_cols.dropna().copy()
df = df.loc[valid.index].reset_index(drop=True)
player_cols = player_cols.loc[valid.index].reset_index(drop=True)

# STEP 4: Add extracted columns back
df = pd.concat([df, player_cols], axis=1)


In [14]:
df.shape

(12944, 108)

In [15]:
# #split the player name and session title columns into columns 
# df[['match_day', 'club_for', 'club_against', 'location', 'league', 'result']] = df['session_title'].str.split('-', n=5, expand=True)
# df[['p_name', 'player_club_', 'player_position']] = df['player_name'].str.split('-', n=2, expand=True)
df = df.drop(['session_title', 'player_name'], axis=1)

# # Apply cleaning to each text column again to catch any new text data
# cat_cols = [col for col in df.columns if df[col].dtype == 'object']
# for col in cat_cols:
#     df[col] = df[col].astype(str).apply(clean_text)

In [16]:
#ensure all entries have the proper 'league' value; fix any that may be interchanged with location or result 
mask = df['league'] != 'League'
df.loc[mask, ['location', 'league']] = df.loc[mask, ['league', 'location']].values
df.loc[df['league'] == 'Home', 'league'] = 'League'
df = df.drop('league', axis=1)

In [17]:
#ensure proper location values; switch any interchanged and fill any missed one 
df.loc[df['location'] == 'Draw', ['location', 'result']] = ['Home', 'Draw']
df.loc[df['location'] == 'Upl', 'location'] = np.nan
df.loc[df['location'] == 'Way', ['location', 'result']] = ['Away', np.nan]
df.loc[df['location'].isna(), 'location'] = 'Home' # it was only maroons (md16) without a location - filled manually after google search 

In [18]:
df.loc[df['result'].isna(), 'result'] = 'Win' # it was only soltilo (md29) without a result - filled manually after a google search

In [19]:
df.shape

(12944, 105)

In [20]:
# list of standard UPL club names
standard_clubs = [
    "KCCA FC", "BUL FC", "URA FC", "Vipers SC", "Express FC", "SC Villa",
    "Maroons FC", "Wakiso Giants FC", "Soltilo Bright Stars FC", "Police FC",
    "UPDF FC", "NEC FC", "Mbarara City FC", "Kitara FC",'Lugazi FC','Mbale Heroes FC'
]
def normalize_name(name):
    # Remove spaces, punctuation, and 'fc', 'sc', etc.
    name = name.lower()
    name = re.sub(r'[^a-z]', '', name)  # keep only letters
    name = name.replace('fc', '').replace('sc', '')
    return name

def best_match(name, club_list, min_score=0.6):
    name_clean = name.strip().lower().replace('.', '')
    norm_name = normalize_name(name_clean)
    # 1. Normalized exact match
    for club in club_list:
        if norm_name == normalize_name(club):
            return club
    # 2. Normalized substring match
    for club in club_list:
        club_norm = normalize_name(club)
        if norm_name in club_norm or club_norm in norm_name:
            return club
    # 3. Token overlap (as before)
    name_tokens = set(name_clean.split())
    best = None
    best_score = 0
    for club in club_list:
        club_tokens = set(club.strip().lower().replace('.', '').split())
        score = len(name_tokens & club_tokens) / max(len(club_tokens), 1)
        if score > best_score:
            best = club
            best_score = score
    return best if best_score >= min_score else name

# Apply to all columns with club names 
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(lambda x: best_match(x, standard_clubs))


In [21]:
df.shape

(12944, 105)

In [22]:
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        df[col] = df[col].replace('Solitilo Bright Stars', 'Soltilo Bright Stars FC')
        df[col] = df[col].replace('Soltito Bright Stars', 'Soltilo Bright Stars FC')
        df[col] = df[col].replace('Solitilo', 'Soltilo Bright Stars FC')
        df[col] = df[col].replace('Soliyilo', 'Soltilo Bright Stars FC')
        df[col] = df[col].replace('Lugzi', 'Lugazi FC')

In [23]:
# Drop rows where entries of  any of the specified columns are not in standard_clubs -remove female clubs
cols_to_check = ['club_for', 'club_against', 'player_club_']
mask = df[cols_to_check].apply(lambda x: all(val in standard_clubs for val in x), axis=1)
df = df[mask]

In [24]:
df.shape

(12914, 105)

In [25]:
# df[df['club_for'] != df['player_club_']][['p_name','club_for', 'player_club_']]
#leave both the 'club_for' and 'player_club_' columnns , could be useful for transfer analysis

In [26]:
missing_positions={
    'Saidi Mayanja':'CM',
    'Ashraf Mugume':'CM',
    'Musitafa Mujuzi':'CB',
    'Bright Anukani':'AM',
    'Kiza Arafat':'AM',
    'Joel Sserunjogi':'DM',
    'Katenga Etienne Openga':'LW',
    'Hassan Muhamud':'CB',
    'Derrick Paul':'LW',
    'Emmanuel Anyama':'CF',
    'Abubaker Mayanja':'CF',
    'Isa Mubiru':'LB',
    'Julius Poloto':'MD',
    'Peter Magambo':'DM',
}

In [27]:
# Replace None player_position using missing_positions and p_name
mask = df['player_position'] == 'None'
df.loc[mask, 'player_position'] = df.loc[mask, 'p_name'].map(missing_positions).fillna('None')

In [28]:
df['player_position'] = df['player_position'].str.lower()

In [29]:
df['general_position'] = df['player_position'].map({
            'gk': 'goalkeeper', 'df': 'defender', 'mf': 'midfielder', 'am': 'midfielder',
            'fw': 'forward', 'rb': 'defender', 'cb': 'defender', 'lb': 'defender',
            'rw': 'forward', 'lw': 'forward', 'cm': 'midfielder', 'dm': 'midfielder',
            'cd':'defender','fwd':'forward','dmc':'midfielder','amc':'midfielder','dc':'defender',
            'cf':'forward','mc':'midfielder','lcb':'defender','md':'midfielder'
        })

In [30]:
df['general_position'].unique()

array(['midfielder', 'defender', 'forward', 'goalkeeper'], dtype=object)

In [31]:
df = df[df['player_position'] != 'gk']

In [32]:
df.shape

(12100, 106)

In [33]:
# Apply cleaning to each text column again to catch any new text data
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)
    df[col] = df[col].astype('category')

In [34]:
zero_frac = (df == 0).mean().sort_values(ascending=False)
sparse = zero_frac[zero_frac > 0.95]
sparse.index.tolist()

# #drop these columns(more than 95% of their values are 0) from the dataset
df = df.drop(columns=sparse.index.tolist())

In [35]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns#all numeric columns
num_cols = ['duration','distance_km','sprint_distance_m','power_plays','energy_kcal','impacts',
                   'player_load','top_speed_kmh','distance_per_min_mmin','power_score_wkg','work_ratio','max_acceleration_mss','max_deceleration_mss']# numeric columns of high interest

In [36]:
#Identify the columns with missing data
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(f'Number of columns with missing data: {len(missing_counts[missing_counts > 0])}')
# Check for duplicate rows in the dataset
duplicate_rows = df.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")
df = df[~duplicate_rows]  # Remove duplicate rows

df = df.drop(columns=['split_start_time', 'split_end_time'])


Number of columns with missing data: 0
Number of duplicate rows: 1


In [37]:
df.shape

(12099, 91)

In [38]:
raw_counts = df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Kcca Fc'].sort_values('match_day')

C:\Users\Travail\AppData\Local\Temp\ipykernel_17892\1146139345.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  raw_counts = df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')


,club_for,match_day,raw_unique_players
60,Kcca Fc,Md1,12
61,Kcca Fc,Md10,14
62,Kcca Fc,Md11,15
63,Kcca Fc,Md12,15
64,Kcca Fc,Md13,12
65,Kcca Fc,Md14,13
66,Kcca Fc,Md15,12
67,Kcca Fc,Md16,16
68,Kcca Fc,Md17,15
69,Kcca Fc,Md18,13


In [39]:
def describe_numeric_columns(df):
    n_desc = df[num_cols].describe().T
    n_desc = n_desc.drop(['25%', '75%'], axis=1)
    n_desc = n_desc.rename(columns={'50%': 'median'})
    display(n_desc.T[num_cols].T)

def describe_categorical_columns(df):
    cat_vars = df.select_dtypes(include='category').columns
    c_desc = df[cat_vars].describe().T
    c_desc['%age'] = c_desc['freq'] / len(df) * 100
    display(c_desc)
    
def plot_numerical_distributions(df):
    plt.figure(figsize=(16, 8))
    for i, col in enumerate(num_cols):
        plt.subplot(4, 4, i + 1)
        sns.histplot(df[col], bins=30, kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
    plt.tight_layout()



In [40]:
half2_df = df[df['split_name'] == '2Nd Half']
half2_df = half2_df.drop('split_name', axis=1)

half1_df = df[df['split_name'] == '1St Half']
half1_df = half1_df.drop('split_name', axis=1)

In [41]:
merge_keys = ['p_name', 'club_for', 'club_against', 'player_club_','match_day','general_position','player_position','result','location']


half1_df_rn = half1_df.rename(columns=lambda x: x + '_H1' if x not in merge_keys else x)
half2_df_rn = half2_df.rename(columns=lambda x: x + '_H2' if x not in merge_keys else x)

df_merged = pd.merge(half1_df_rn, half2_df_rn, on=merge_keys, how='outer')


df_combined = df_merged[merge_keys].copy()

# Numerical Variables of high interest - consider sum
for col in num_cols:
    if col not in ['top_speed_kmh', 'distance_per_min_mmin','max_acceleration_mss','max_deceleration_mss']:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        # Use .fillna(0) to handle missing halves
        df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)

# Top Speed - consider max
df_combined['top_speed_kmh'] = df_merged[['top_speed_kmh_H1', 'top_speed_kmh_H2']].max(axis=1)
# Max Acceleration - consider max
df_combined['max_acceleration_mss'] = df_merged[['max_acceleration_mss_H1', 'max_acceleration_mss_H2']].max(axis=1)
# Max Deceleration - consider max
df_combined['max_deceleration_mss'] = df_merged[['max_deceleration_mss_H1', 'max_deceleration_mss_H2']].max(axis=1)

# distance per minute - recalculate from sum
df_combined['distance_per_min_mmin'] = (df_combined['distance_km'] *1000) / df_combined['duration']

#all other numerical variables - consider sum
# Columns in numeric_cols that are not in num_cols
for col in numeric_cols.difference(num_cols):
    if col not in merge_keys:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        # Only sum if both columns exist in df_merged
        if h1 in df_merged.columns and h2 in df_merged.columns:
            df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)


In [42]:
match_df = df_combined

In [43]:
dur_counts = match_df.groupby(['club_for', 'match_day'])['duration'].mean().reset_index(name='avg_duration')
dur_counts[dur_counts['club_for'] == 'Sc Villa'].sort_values('match_day')

raw_counts = match_df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Sc Villa'].sort_values('match_day')

C:\Users\Travail\AppData\Local\Temp\ipykernel_17892\3209795986.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dur_counts = match_df.groupby(['club_for', 'match_day'])['duration'].mean().reset_index(name='avg_duration')
C:\Users\Travail\AppData\Local\Temp\ipykernel_17892\3209795986.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  raw_counts = match_df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')


,club_for,match_day,raw_unique_players
300,Sc Villa,Md1,13
301,Sc Villa,Md10,13
302,Sc Villa,Md11,14
303,Sc Villa,Md12,11
304,Sc Villa,Md13,10
305,Sc Villa,Md14,10
306,Sc Villa,Md15,8
307,Sc Villa,Md16,8
308,Sc Villa,Md17,0
309,Sc Villa,Md18,9


In [44]:
# Thresholds
DIST_THRESH = 2   # players should cover at least 2km in a session to count
DURATION_THRESH = 60  # players should be active for at least 60 minutes in a session    


# Extend your active_session flag:
match_df['active_session'] = (
    (match_df['distance_km']    >= DIST_THRESH)  
    & (match_df['duration']  >= DURATION_THRESH)   
)

match_df = match_df[match_df['active_session'] == True]  # Filter main df for active sessions


In [45]:
match_df = match_df.drop('active_session', axis=1)  # Drop the active_session column after filtering

In [46]:
# Distance IQR
q1_dist = match_df['distance_km'].quantile(0.25)
q3_dist = match_df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_bound_dist = q1_dist - 1.5 * iqr_dist
upper_bound_dist = q3_dist + 1.5 * iqr_dist

# Duration IQR
q1_dur = match_df['duration'].quantile(0.25)
q3_dur = match_df['duration'].quantile(0.75)
iqr_dur = q3_dur - q1_dur
lower_bound_dur = q1_dur - 1.5 * iqr_dur
upper_bound_dur = q3_dur + 1.5 * iqr_dur

# Filter out outliers
match_df = match_df[
    (match_df['distance_km'] >= lower_bound_dist) & (match_df['distance_km'] <= upper_bound_dist) &
    (match_df['duration']    >= lower_bound_dur)  & (match_df['duration']    <= upper_bound_dur)
]


In [47]:
print(f"Duration bounds: {lower_bound_dur:.2f} to {upper_bound_dur:.2f}")
print(f"Distance bounds: {lower_bound_dist:.2f} to {upper_bound_dist:.2f}")


Duration bounds: 85.45 to 109.98
Distance bounds: 2.37 to 14.81


In [48]:
match_df['duration'].describe()

count    3988.000000
mean       98.113503
std         4.590174
min        85.483333
25%        95.566667
50%        97.983333
75%       100.833333
max       109.616667
Name: duration, dtype: float64

In [49]:
describe_categorical_columns(match_df)

,count,unique,top,freq,%age
p_name,3988,384,Emmanuel Obua,27,0.677031
club_for,3988,16,Mbarara City Fc,336,8.425276
club_against,3988,16,Wakiso Giants Fc,280,7.021063
player_club_,3988,16,Nec Fc,338,8.475426
match_day,3988,30,Md7,165,4.137412
general_position,3988,3,Defender,1573,39.44333
player_position,3988,20,Cb,586,14.694082
result,3988,3,Win,1542,38.665998
location,3988,2,Away,2053,51.479438


In [50]:
raw_counts = match_df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Maroons Fc'].sort_values('match_day')


C:\Users\Travail\AppData\Local\Temp\ipykernel_17892\58886186.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  raw_counts = match_df.groupby(['club_for', 'match_day'])['p_name'].nunique().reset_index(name='raw_unique_players')


,club_for,match_day,raw_unique_players
150,Maroons Fc,Md1,12
151,Maroons Fc,Md10,0
152,Maroons Fc,Md11,12
153,Maroons Fc,Md12,10
154,Maroons Fc,Md13,11
155,Maroons Fc,Md14,12
156,Maroons Fc,Md15,12
157,Maroons Fc,Md16,11
158,Maroons Fc,Md17,0
159,Maroons Fc,Md18,0


In [51]:
dur_counts = match_df.groupby(['club_for', 'match_day'])['duration'].mean().reset_index(name='avg_duration')
dur_counts[dur_counts['club_for'] == 'Vipers Sc'].sort_values('match_day')

C:\Users\Travail\AppData\Local\Temp\ipykernel_17892\1055080764.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dur_counts = match_df.groupby(['club_for', 'match_day'])['duration'].mean().reset_index(name='avg_duration')


,club_for,match_day,avg_duration
420,Vipers Sc,Md1,97.677778
421,Vipers Sc,Md10,100.622222
422,Vipers Sc,Md11,98.066667
423,Vipers Sc,Md12,103.462963
424,Vipers Sc,Md13,103.183333
425,Vipers Sc,Md14,107.950000
426,Vipers Sc,Md15,95.958333
427,Vipers Sc,Md16,92.766667
428,Vipers Sc,Md17,97.983333
429,Vipers Sc,Md18,95.725000


In [52]:
# Save the cleaned and processed DataFrame to a new CSV file
# match_df.to_csv('UPL25_matches2.csv',index=False)